# 모듈과 타입힌트

함수를 별도 모듈로 나누고, 타입힌트의 역할과 한계를 어떻게 확인할까요?

## 실행 환경 준비

Python 3.12 표준 라이브러리만 사용합니다. 모듈 파일은 임시 폴더에 만들고 하위 프로세스에서 불러옵니다.

In [1]:
# 모듈과 실행 파일을 저장할 실습 전용 임시 폴더를 준비합니다.
from pathlib import Path
from subprocess import run
from sys import executable
from tempfile import TemporaryDirectory

workspace = TemporaryDirectory(prefix="ch02_07_")
module_dir = Path(workspace.name)
print("모듈 실습 폴더 준비 완료")

모듈 실습 폴더 준비 완료


## 기본 실행

`pricing.py`는 함수를 담은 모듈입니다. 실행 파일은 `from pricing import add_tax`로 필요한 이름만 가져옵니다.

In [2]:
# 계산 함수를 모듈에 저장하고 별도 파일에서 import하여 호출합니다.
module_source = '''def add_tax(price: int, rate: float) -> int:
    return round(price * (1 + rate))
'''
runner_source = '''from pricing import add_tax

print(add_tax(10000, 0.1))
'''
(module_dir / "pricing.py").write_text(module_source, encoding="utf-8")
(module_dir / "run_price.py").write_text(runner_source, encoding="utf-8")
completed = run([executable, "run_price.py"], cwd=module_dir, capture_output=True, text=True, check=True)
print("세금 포함 가격:", completed.stdout.strip())

세금 포함 가격: 11000


## 중간 결과 관찰

함수의 `__annotations__`에는 개발자가 적은 입력과 반환 타입이 저장됩니다. 이 정보는 함수 설명과 정적 검사에 사용됩니다.

In [3]:
# 타입힌트 정보와 실제 함수 반환값을 함께 확인합니다.
def label_price(price: int) -> str:
    return f"{price:,}원"

print(label_price.__annotations__)
print(label_price(12000))

{'price': <class 'int'>, 'return': <class 'str'>}
12,000원


## 값 하나를 바꿔 비교

같은 함수에서 가격 값만 바꾸면 반환 문자열이 어떻게 달라지는지 확인합니다.

In [4]:
# 함수에 전달하는 가격만 바꾸어 표시 결과를 비교합니다.
print("12,000 입력:", label_price(12000))
print("25,000 입력:", label_price(25000))
assert label_price(25000) == "25,000원"

12,000 입력: 12,000원
25,000 입력: 25,000원


## 실패 재현

타입힌트를 적어도 Python은 호출 시 값을 자동 검사하지 않습니다. `int`라고 표시한 함수에 문자열을 주면 문자열 반복이라는 잘못된 결과가 나옵니다.

In [5]:
# 타입힌트가 문자열 입력을 자동으로 거부하지 않는지 확인합니다.
def double_total(total: int) -> int:
    return total * 2

wrong = double_total("1200")  # 타입힌트와 다른 입력
print("잘못된 결과:", wrong)
assert wrong == "12001200"

잘못된 결과: 12001200


## 수정

외부에서 받은 문자열을 함수 호출 전에 정수로 변환합니다. 같은 `"1200"` 입력이 숫자 계산 결과 2400으로 바뀝니다.

In [6]:
# 입력 문자열을 정수로 바꾸어 숫자 곱셈 결과를 얻습니다.
raw_total = "1200"
validated_total = int(raw_total)
correct = double_total(validated_total)
print("수정된 결과:", correct)
assert correct == 2400

수정된 결과: 2400
